# **Import et config**

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from biomae.paths import get_project_root

BASE_DIR = get_project_root()
WORK_DIR = BASE_DIR / "travail"
EMBEDDINGS_DIR = BASE_DIR / "embeddings_data"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)


# **Verifie la cohérence des labels des 2 cameras**

# **Extraction des embbeddings du modele YOLO**

In [ ]:
class FeatureExtractorMulti:
    def __init__(self, model_path):
        self.yolo = YOLO(model_path)
        self.model = self.yolo.model
        self.target_layer = self.model.model[9] 
        self.features = None
        self.target_layer.register_forward_hook(self.hook_fn)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))

    def hook_fn(self, module, input, output):
        self.features = output

    def extract_all(self, img_path):
        """
        Renvoie TOUS les gammares détectés, triés de Gauche à Droite.
        Retourne : liste de (pred_class, embedding, bbox)
        """
        # Inférence
        results = self.yolo.predict(img_path, conf=0.25, verbose=False)
        
        if len(results[0].boxes) == 0:
            return []

        detections = []
        
        # Le hook a capturé la feature map GLOBALE de l'image.
        # Pour faire ça parfaitement en multi-objets, il faudrait "ROI Align".
        # MAIS pour le MVP, on utilise l'approximation suivante :
        # On suppose que l'embedding global contient l'info dominante. 
        # Si tu as plusieurs gammares, cette méthode "Global Pool" est imparfaite 
        # (elle mélange les caractéristiques des deux).
        
        # --- SOLUTION AVANCÉE : Crop & Re-Pass ---
        # Pour avoir un embedding propre par gammare, on doit découper et repasser dans le réseau.
        # C'est la seule façon propre de gérer le multi-objets sans ROI Align complexe.
        
        img_full = results[0].orig_img # L'image chargée par OpenCV
        
        # On récupère toutes les boîtes
        boxes = results[0].boxes
        
        for i, box in enumerate(boxes):
            # Coordonnées pixel pour le crop
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            conf = float(box.conf)
            cls = int(box.cls)
            xywh = box.xywhn[0].cpu().numpy() # Pour le matching label plus tard
            
            # Sécurité bords
            h, w, _ = img_full.shape
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            
            if (x2-x1) < 5 or (y2-y1) < 5: continue # Trop petit
            
            # 1. CROP (Découpage)
            crop = img_full[y1:y2, x1:x2]
            
            # 2. RE-PASSAGE DU CROP DANS LE BACKBONE
            # On fait une inférence sur le petit bout d'image pour avoir SON embedding pur
            # On utilise embed() natif s'il marche, sinon notre hook sur le crop
            # Ici on réutilise notre hook en faisant predict sur le crop
            _ = self.yolo.predict(crop, verbose=False) # Déclenche le hook sur le crop
            
            # 3. Extraction Embedding
            pooled = self.avg_pool(self.features)
            emb_vec = pooled.flatten().cpu().detach().numpy()
            
            detections.append({
                'cls': cls,
                'emb': emb_vec,
                'box': xywh,
                'center_x': xywh[0] # Pour le tri
            })
            
        # TRI GÉOMÉTRIQUE : On trie les détections par leur position X (gauche -> droite)
        # Comme ça, le 1er gammare de A correspondra au 1er gammare de B
        detections.sort(key=lambda x: x['center_x'])
        
        return detections

extractor = FeatureExtractorMulti(MODEL_PATH)
print("Extracteur Multi-Objets (Crop-based) prêt !")

# **Appariement des images et réorganisation**

In [ ]:
import os
import glob
from tqdm.notebook import tqdm

dataset_features = []
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')

for split in ['train', 'val']:
    img_dir = os.path.join(WORK_DIR, split, 'images')
    if not os.path.exists(img_dir): continue

    all_files = [f for f in os.listdir(img_dir) if f.lower().endswith(valid_extensions)]
    unique_ids = set([os.path.splitext(f)[0].split('_')[-1] for f in all_files])
    
    print(f"Split '{split}'...")
    
    count_pairs = 0
    
    for sp_id in tqdm(unique_ids):
        cand_A = glob.glob(os.path.join(img_dir, f"*_A_{sp_id}.*"))
        cand_B = glob.glob(os.path.join(img_dir, f"*_B_{sp_id}.*"))
        
        if not cand_A or not cand_B: continue
        path_A, path_B = cand_A[0], cand_B[0]
        
        # 1. On récupère TOUTES les détections triées
        dets_A = extractor.extract_all(path_A)
        dets_B = extractor.extract_all(path_B)
        
        # 2. Logique d'Appariement (Pairing)
        # On ne peut former une paire que si on a un candidat des deux côtés.
        # On prend le minimum (si A en voit 3 et B en voit 2, on ne peut faire que 2 paires sûres)
        num_pairs = min(len(dets_A), len(dets_B))
        
        for i in range(num_pairs):
            d_A = dets_A[i] # Le i-ème gammare en partant de la gauche sur A
            d_B = dets_B[i] # Le i-ème gammare en partant de la gauche sur B
            
            # --- MATCHING LABEL ---
            # On cherche le vrai label dans le fichier txt de A pour CETTE boîte
            label_path = path_A.replace('images', 'labels')
            label_path = os.path.splitext(label_path)[0] + '.txt'
            
            true_label = -1
            if os.path.exists(label_path):
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                    best_iou = 0
                    for line in lines:
                        parts = list(map(float, line.split()))
                        cls_gt = int(parts[0])
                        box_gt = parts[1:5]
                        
                        # On compare avec la boîte de A
                        iou = calculate_iou(d_A['box'], box_gt)
                        if iou > 0.5 and iou > best_iou: # Seuil IoU 0.5
                            best_iou = iou
                            true_label = cls_gt
            
            if true_label != -1:
                dataset_features.append({
                    'id': f"{sp_id}_{i}", # ID unique par individu (0000_0, 0000_1...)
                    'emb_A': d_A['emb'],
                    'emb_B': d_B['emb'],
                    'label': true_label,
                    'pred_A': d_A['cls'],
                    'pred_B': d_B['cls'],
                    'split': split
                })
                count_pairs += 1

    print(f" -> {count_pairs} paires (individus) extraites.")

# Sauvegarde
with open(os.path.join(EMBEDDINGS_DIR, "dataset_features.pkl"), 'wb') as f:
    pickle.dump(dataset_features, f)
print("Sauvegarde terminée.")

# **Appariement des images et réorganisation**

In [ ]:
import pickle

save_path = os.path.join(EMBEDDINGS_DIR, "dataset_features.pkl")

with open(save_path, 'wb') as f:
    pickle.dump(dataset_features, f)

print(f"Dataset sauvegardé sous : {save_path}")

# Vérification de la taille d'un vecteur
if len(dataset_features) > 0:
    dim = dataset_features[0]['emb_A'].shape[0]
    print(f"Dimension du vecteur d'embedding : {dim}")
    print("Exemple de données :", dataset_features[0].keys())

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import seaborn as sns
PKL_PATH = os.path.join(BASE_DIR, "embeddings_data", "dataset_features.pkl")

# 1. Chargement
if not os.path.exists(PKL_PATH):
    print("ERREUR : Fichier pickle introuvable !")
else:
    with open(PKL_PATH, 'rb') as f:
        data = pickle.load(f)
    print(f"Chargement de {len(data)} paires de données.")

    # 2. Préparation des données pour l'analyse
    # On va empiler tous les vecteurs (A et B) pour voir comment ils se distribuent
    vectors = []
    labels = []
    sources = [] # 'Vue A' ou 'Vue B'
    pairs_similarity = [] # Pour stocker la distance entre A et B

    class_names = {0: 'Male', 1: 'Femelle', 2: 'Indetermine', 3: 'Couple'}

    for item in data:
        vec_A = item['emb_A']
        vec_B = item['emb_B']
        label = class_names.get(item['label'], str(item['label']))
        
        # Ajout aux listes
        vectors.append(vec_A)
        labels.append(label)
        sources.append('Vue A')
        
        vectors.append(vec_B)
        labels.append(label)
        sources.append('Vue B')
        
        # Calcul de similarité (Cosinus) entre A et B pour cet individu
        # Plus c'est proche de 1.0, plus les vues sont cohérentes
        sim = np.dot(vec_A, vec_B) / (np.linalg.norm(vec_A) * np.linalg.norm(vec_B))
        pairs_similarity.append(sim)

    vectors = np.array(vectors)
    print(f"\n--- Analyse de cohérence A/B ---")
    print(f"Similarité moyenne entre Vue A et Vue B : {np.mean(pairs_similarity):.3f}")
    print("(Une valeur > 0.5 est excellente, cela veut dire que le modèle 'comprend' que c'est le même objet sous deux angles)")

    # 3. Visualisation PCA (Projection 2D rapide)
    print("\nCalcul de la projection 2D (PCA)...")
    pca = PCA(n_components=2)
    vec_2d = pca.fit_transform(vectors)

    # Création du graphique
    plt.figure(figsize=(10, 8))
    sns.scatterplot(
        x=vec_2d[:,0], 
        y=vec_2d[:,1], 
        hue=labels, 
        style=sources,
        palette="viridis",
        s=60,
        alpha=0.8
    )
    plt.title("Visualisation des Embeddings (Espace Latent du Gammare)")
    plt.xlabel("Composante Principale 1")
    plt.ylabel("Composante Principale 2")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# **Config**

In [ ]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

EMBEDDINGS_PATH = os.path.join(BASE_DIR, "embeddings_data", "dataset_features.pkl")

# Paramètres du modèle
HIDDEN_SIZE = 256
DROPOUT_RATE = 0.5
LEARNING_RATE = 0.001
BATCH_SIZE = 16
EPOCHS = 50

# Détection du GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entraînement sur : {device}")

# **Dataset PyTorch**

In [ ]:
class PairEmbeddingDataset(Dataset):
    def __init__(self, pkl_path, split_type='train'):
        """
        split_type : 'train' ou 'val'
        """
        with open(pkl_path, 'rb') as f:
            raw_data = pickle.load(f)
        
        # Filtrer les données selon le split (train/val)
        self.data = [item for item in raw_data if item['split'] == split_type]
        
        # Vérification des classes
        self.labels = [d['label'] for d in self.data]
        self.num_classes = len(set(self.labels)) if len(self.labels) > 0 else 0
        
        print(f"Split '{split_type}' chargé : {len(self.data)} échantillons.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Conversion Numpy -> Tensor PyTorch
        emb_A = torch.tensor(item['emb_A'], dtype=torch.float32)
        emb_B = torch.tensor(item['emb_B'], dtype=torch.float32)
        label = torch.tensor(item['label'], dtype=torch.long)
        
        # On retourne les embeddings séparés (la concaténation se fera dans le modèle)
        # On pourrait aussi concaténer ici.
        return emb_A, emb_B, label

# Création des Datasets et DataLoaders
train_dataset = PairEmbeddingDataset(EMBEDDINGS_PATH, split_type='train')
val_dataset = PairEmbeddingDataset(EMBEDDINGS_PATH, split_type='val')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True) # on active le drop_last pour avoir un multiple de 8
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Récupération de la dimension d'entrée dynamiquement
if len(train_dataset) > 0:
    sample_A, _, _ = train_dataset[0]
    INPUT_DIM = sample_A.shape[0] # Devrait être 512 ou 256
    NUM_CLASSES = 3 # Force à 4 (Male, Femelle, Indetermine, Couple) pour être sûr
    print(f"Dimension vecteur entrée : {INPUT_DIM}")
    print(f"Nombre de classes : {NUM_CLASSES}")

# **Archi MLP**

In [ ]:
class FusionMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout_rate=0.5):
        super(FusionMLP, self).__init__()
        
        # La taille d'entrée est doublée car on concatène A + B
        self.concat_dim = input_dim * 2 
        
        self.layer1 = nn.Sequential(
            nn.Linear(self.concat_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim), # Stabilise l'apprentissage
            nn.ReLU(),
            nn.Dropout(dropout_rate)    # Évite le par-coeur (Overfitting)
        )
        
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, emb_A, emb_B):
        # 1. Concaténation des vecteurs A et B
        # [Batch, 512] + [Batch, 512] -> [Batch, 1024]
        x = torch.cat((emb_A, emb_B), dim=1)
        
        # 2. Passage dans les couches cachées
        x = self.layer1(x)
        
        # 3. Classification finale
        logits = self.classifier(x)
        return logits

# Initialisation du modèle
model = FusionMLP(INPUT_DIM, HIDDEN_SIZE, NUM_CLASSES, DROPOUT_RATE).to(device)
print(model)

# **Boucle d'Entraînement**

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Pour stocker l'historique
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print("Début de l'entraînement du MLP...")

best_val_acc = 0.0
patience = 10
trigger_times = 0

for epoch in range(EPOCHS):
    # --- TRAIN ---
    model.train()
    running_loss = 0.0
    for emb_A, emb_B, labels in train_loader:
        emb_A, emb_B, labels = emb_A.to(device), emb_B.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(emb_A, emb_B)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(train_loader)
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for emb_A, emb_B, labels in val_loader:
            emb_A, emb_B, labels = emb_A.to(device), emb_B.to(device), labels.to(device)
            outputs = model(emb_A, emb_B)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    epoch_val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(all_labels, all_preds)
    
    # Logs
    history['train_loss'].append(epoch_loss)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Early Stopping et Sauvegarde du meilleur modèle
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        trigger_times = 0
        torch.save(model.state_dict(), os.path.join(BASE_DIR, 'best_fusion_mlp.pth'))
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print(f"Early stopping ! Pas d'amélioration depuis {patience} époques.")
            break

print(f"Entraînement terminé. Meilleure Val Acc: {best_val_acc:.4f}")

# **Comparaison**

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
import pandas as pd

# 1. Chargement du modèle
model.load_state_dict(torch.load(os.path.join(BASE_DIR, 'best_fusion_mlp.pth')))
model.eval()

# 2. Collecte des prédictions
yolo_A_preds = []
yolo_B_preds = []
fusion_preds = []
true_labels = []

# Noms des classes (Assure-toi que l'ordre correspond à ton fichier data.yaml)
class_names = ['Male', 'Femelle', 'Indeter', 'Couple']
# Indices correspondant (pour forcer la matrice 4x4 même si une classe est absente)
class_indices = [0, 1, 2, 3]

print("Génération des statistiques comparatives...")

with torch.no_grad():
    for item in val_dataset.data:
        # Vérité terrain
        true_labels.append(item['label'])
        
        # YOLO Seul (Baseline)
        yolo_A_preds.append(item['pred_A'])
        yolo_B_preds.append(item['pred_B'])
        
        # Fusion MLP
        emb_A = torch.tensor(item['emb_A']).unsqueeze(0).to(device)
        emb_B = torch.tensor(item['emb_B']).unsqueeze(0).to(device)
        output = model(emb_A, emb_B)
        _, pred_fusion = torch.max(output, 1)
        fusion_preds.append(pred_fusion.item())

# --- 3. FONCTION DE CALCUL DES MÉTRIQUES ---
def get_metrics_summary(y_true, y_pred, name):
    # Accuracy globale
    acc = accuracy_score(y_true, y_pred)
    # Précision, Rappel, F1 (Macro = moyenne non pondérée, traite toutes les classes à égalité)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return [name, acc, prec, rec, f1]

# Création du tableau comparatif
metrics_data = []
metrics_data.append(get_metrics_summary(true_labels, yolo_A_preds, "YOLO Vue A"))
metrics_data.append(get_metrics_summary(true_labels, yolo_B_preds, "YOLO Vue B"))
metrics_data.append(get_metrics_summary(true_labels, fusion_preds, "FUSION MLP"))

df_metrics = pd.DataFrame(metrics_data, columns=['Modèle', 'Accuracy', 'Précision (Macro)', 'Rappel (Macro)', 'F1-Score'])
df_metrics.set_index('Modèle', inplace=True)

# Affichage du tableau (x100 pour %)
print("\n=== TABLEAU DES PERFORMANCES (%) ===")
print(df_metrics.multiply(100).round(2))
print("====================================")

# --- 4. VISUALISATION DES MATRICES DE CONFUSION ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Liste des configurations à tracer
plots = [
    (yolo_A_preds, "YOLO Vue A"),
    (yolo_B_preds, "YOLO Vue B"),
    (fusion_preds, "FUSION MLP (Ours)")
]

for i, (preds, title) in enumerate(plots):
    cm = confusion_matrix(true_labels, preds, labels=class_indices)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=class_names, yticklabels=class_names, ax=axes[i])
    axes[i].set_title(title, fontsize=14, fontweight='bold')
    axes[i].set_xlabel('Prédiction')
    if i == 0:
        axes[i].set_ylabel('Vraie Classe')
    else:
        axes[i].set_ylabel('') # Enlever le label Y pour les graphes suivants

plt.tight_layout()
plt.show()

# --- 5. CONCLUSION AUTOMATIQUE ---
gain_acc = df_metrics.loc['FUSION MLP', 'Accuracy'] - max(df_metrics.loc['YOLO Vue A', 'Accuracy'], df_metrics.loc['YOLO Vue B', 'Accuracy'])
gain_f1 = df_metrics.loc['FUSION MLP', 'F1-Score'] - max(df_metrics.loc['YOLO Vue A', 'F1-Score'], df_metrics.loc['YOLO Vue B', 'F1-Score'])


if gain_acc > 0:
    print(f"- La fusion améliore l'Accuracy globale de +{gain_acc*100:.2f} points.")
else:
    print(f"- Accuracy stable (pas de gain global).")

if gain_f1 > 0:
    print(f"- La fusion améliore le F1-Score (équilibre) de +{gain_f1*100:.2f} points.")
    print("  (C'est souvent le signe qu'on récupère mieux les classes difficiles comme 'Couple' ou 'Indeter')")